In [1]:
import requests
import time
import io
from datetime import datetime, timezone, timedelta
import pandas as pd


In [2]:
API_TOKEN = "y0__xDe_Im4BBiirjkg2Zzs_xP3srDDcLIeER6ns2I71_wDiFmYXg"
COUNTER_ID = 103580753
BASE_URL = "https://api-metrika.yandex.net"
HEADERS = {
    "Authorization": f"OAuth {API_TOKEN}"
}


In [3]:
VISITS_FIELDS = [
    "ym:s:visitID",
    "ym:s:isNewUser",
    "ym:s:counterUserIDHash",
    "ym:s:clientID",
    "ym:s:regionCountry",
    "ym:s:regionCity",
    "ym:s:regionCountryID",
    "ym:s:regionCityID",
    "ym:s:operatingSystem",
    "ym:s:deviceCategory",
    "ym:s:startURL",
    "ym:s:pageViews",
    "ym:s:lastTrafficSource",
    "ym:s:dateTimeUTC"
]


In [4]:
create_url = f"{BASE_URL}/management/v1/counter/{COUNTER_ID}/logrequests"

payload = {
    "date1": "2025-07-01",
    "date2": "2025-12-31",
    "source": "visits",
    "fields": ",".join(VISITS_FIELDS)
}

response = requests.post(create_url, headers=HEADERS, data=payload)
response.raise_for_status()

request_id = response.json()["log_request"]["request_id"]
print("request_id:", request_id)


request_id: 50760304


In [5]:
status_url = f"{BASE_URL}/management/v1/counter/{COUNTER_ID}/logrequest/{request_id}"
while True:
    status_response = requests.get(status_url, headers=HEADERS)
    status_response.raise_for_status()
    
    status = status_response.json()["log_request"]["status"]
    print("Status:", status)
    
    if status == "processed":
        break
    elif status == "failed":
        raise Exception("Log_request_failed")
    
    time.sleep(5)
    

Status: created
Status: created
Status: created
Status: created
Status: created
Status: created
Status: created
Status: created
Status: created
Status: processed


In [6]:
parts_url = f"{BASE_URL}/management/v1/counter/{COUNTER_ID}/logrequest/{request_id}/parts"
parts_response = requests.get(status_url, headers=HEADERS)
parts_response.raise_for_status()
parts = parts_response.json()["log_request"]["parts"]
print("parts:", parts)


parts: [{'part_number': 0, 'size': 10320720}]


In [7]:
part_number_url = f"{BASE_URL}/management/v1/counter/{COUNTER_ID}/logrequest/{request_id}/part/0/download"
part_number_response = requests.get(part_number_url, headers=HEADERS)
print(part_number_response)


<Response [200]>


In [8]:
visits_df = pd.read_csv(io.BytesIO(part_number_response.content), delimiter='\t')

moscow_tz = timezone(timedelta(hours=3))
visits_df["updated_at"] = datetime.now(moscow_tz)
visits_df


,ym:s:visitID,ym:s:isNewUser,ym:s:counterUserIDHash,ym:s:clientID,ym:s:regionCountry,ym:s:regionCity,ym:s:regionCountryID,ym:s:regionCityID,ym:s:operatingSystem,ym:s:deviceCategory,ym:s:startURL,ym:s:pageViews,ym:s:lastTrafficSource,ym:s:dateTimeUTC,updated_at
0,6096107976723005757,0,14271926282808846030,1759897989270675974,Uzbekistan,Tashkent,171,10335,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,1,direct,2025-10-16 06:40:06,2026-01-20 00:46:16.382475+03:00
1,7480221433098666083,0,5677847916278433233,1758919525441860732,Russia,NaN,225,0,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/C...,2,direct,2025-12-16 09:19:40,2026-01-20 00:46:16.382475+03:00
2,5251787943408304163,1,3320593644407048834,1757365181547967994,Russia,Kazan,225,43,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,2,direct,2025-09-08 23:59:41,2026-01-20 00:46:16.382475+03:00
3,5166047600572104835,0,16290427431084818553,1754617191898040015,United States,NaN,84,0,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,1,direct,2025-09-05 05:08:27,2026-01-20 00:46:16.382475+03:00
4,5553884038399000732,0,422698621777490084,1756757943885508675,Russia,Saint Petersburg,225,2,android_15,2,https://halltape.github.io/HalltapeRoadmapDE/,1,referral,2025-09-22 08:06:25,2026-01-20 00:46:16.382475+03:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57735,5633456023022075906,0,10456232782235392117,17588165399370650,NaN,NaN,0,0,mac_os_catalina,1,https://halltape.github.io/HalltapeRoadmapDE/,2,direct,2025-09-25 20:25:29,2026-01-20 00:46:16.382475+03:00
57736,7468454789294850073,0,11232044201172113103,1758815199622178988,Netherlands,NaN,118,0,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,2,internal,2025-12-15 20:33:18,2026-01-20 00:46:16.382475+03:00
57737,5403981651651330106,1,11059100474524680321,1757945754206369907,Russia,Moscow,225,213,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/,13,referral,2025-09-15 17:15:54,2026-01-20 00:46:16.382475+03:00
57738,4451620541472178406,1,2557959602701053705,1754312785226181066,Russia,Lipetsk,225,9,linux,1,https://halltape.github.io/HalltapeRoadmapDE/DWH/,5,direct,2025-08-04 16:06:24,2026-01-20 00:46:16.382475+03:00


In [9]:
visits_df.columns = [
    col if i > 13 else col[5:]
    for i, col in enumerate(visits_df.columns)
]
visits_df


,visitID,isNewUser,counterUserIDHash,clientID,regionCountry,regionCity,regionCountryID,regionCityID,operatingSystem,deviceCategory,startURL,pageViews,lastTrafficSource,dateTimeUTC,updated_at
0,6096107976723005757,0,14271926282808846030,1759897989270675974,Uzbekistan,Tashkent,171,10335,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,1,direct,2025-10-16 06:40:06,2026-01-20 00:46:16.382475+03:00
1,7480221433098666083,0,5677847916278433233,1758919525441860732,Russia,NaN,225,0,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/C...,2,direct,2025-12-16 09:19:40,2026-01-20 00:46:16.382475+03:00
2,5251787943408304163,1,3320593644407048834,1757365181547967994,Russia,Kazan,225,43,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,2,direct,2025-09-08 23:59:41,2026-01-20 00:46:16.382475+03:00
3,5166047600572104835,0,16290427431084818553,1754617191898040015,United States,NaN,84,0,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,1,direct,2025-09-05 05:08:27,2026-01-20 00:46:16.382475+03:00
4,5553884038399000732,0,422698621777490084,1756757943885508675,Russia,Saint Petersburg,225,2,android_15,2,https://halltape.github.io/HalltapeRoadmapDE/,1,referral,2025-09-22 08:06:25,2026-01-20 00:46:16.382475+03:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57735,5633456023022075906,0,10456232782235392117,17588165399370650,NaN,NaN,0,0,mac_os_catalina,1,https://halltape.github.io/HalltapeRoadmapDE/,2,direct,2025-09-25 20:25:29,2026-01-20 00:46:16.382475+03:00
57736,7468454789294850073,0,11232044201172113103,1758815199622178988,Netherlands,NaN,118,0,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,2,internal,2025-12-15 20:33:18,2026-01-20 00:46:16.382475+03:00
57737,5403981651651330106,1,11059100474524680321,1757945754206369907,Russia,Moscow,225,213,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/,13,referral,2025-09-15 17:15:54,2026-01-20 00:46:16.382475+03:00
57738,4451620541472178406,1,2557959602701053705,1754312785226181066,Russia,Lipetsk,225,9,linux,1,https://halltape.github.io/HalltapeRoadmapDE/DWH/,5,direct,2025-08-04 16:06:24,2026-01-20 00:46:16.382475+03:00


In [10]:
visits_df.to_csv('./tmp/data_visits.csv', sep = '\t', index = False)


In [11]:
HITS_FIELDS = [
    "ym:pv:watchID",
    "ym:pv:pageViewID",
    "ym:pv:counterUserIDHash",
    "ym:pv:clientID",
    "ym:pv:regionCountry",
    "ym:pv:regionCity",
    "ym:pv:regionCountryID",
    "ym:pv:regionCityID",
    "ym:pv:operatingSystem",
    "ym:pv:deviceCategory",
    "ym:pv:URL",
    "ym:pv:productEventType",
    "ym:pv:lastTrafficSource",
    "ym:pv:dateTime"
]


In [12]:
create_url = f"{BASE_URL}/management/v1/counter/{COUNTER_ID}/logrequests"

payload = {
    "date1": "2025-07-01",
    "date2": "2025-12-31",
    "source": "hits",
    "fields": ",".join(HITS_FIELDS)
}

response = requests.post(create_url, headers=HEADERS, data=payload)
response.raise_for_status()

request_id = response.json()["log_request"]["request_id"]
print("request_id:", request_id)

request_id: 50760319


In [13]:
status_url = f"{BASE_URL}/management/v1/counter/{COUNTER_ID}/logrequest/{request_id}"
while True:
    status_response = requests.get(status_url, headers=HEADERS)
    status_response.raise_for_status()
    
    status = status_response.json()["log_request"]["status"]
    print("Status:", status)
    
    if status == "processed":
        break
    elif status == "failed":
        raise Exception("Log_request_failed")
    
    time.sleep(5)
    

Status: created
Status: created
Status: created
Status: processed


In [14]:
parts_url = f"{BASE_URL}/management/v1/counter/{COUNTER_ID}/logrequest/{request_id}/parts"
parts_response = requests.get(status_url, headers=HEADERS)
parts_response.raise_for_status()
parts = parts_response.json()["log_request"]["parts"]
print("parts:", parts)


parts: [{'part_number': 0, 'size': 38870750}]


In [15]:
part_number_url = f"{BASE_URL}/management/v1/counter/{COUNTER_ID}/logrequest/{request_id}/part/0/download"
part_number_response = requests.get(part_number_url, headers=HEADERS)
print(part_number_response)

<Response [200]>


In [16]:
hits_df = pd.read_csv(io.BytesIO(part_number_response.content), delimiter='\t')

moscow_tz = timezone(timedelta(hours=3))
hits_df["updated_at"] = datetime.now(moscow_tz)
hits_df

,ym:pv:watchID,ym:pv:pageViewID,ym:pv:counterUserIDHash,ym:pv:clientID,ym:pv:regionCountry,ym:pv:regionCity,ym:pv:regionCountryID,ym:pv:regionCityID,ym:pv:operatingSystem,ym:pv:deviceCategory,ym:pv:URL,ym:pv:productEventType,ym:pv:lastTrafficSource,ym:pv:dateTime,updated_at
0,7.467877e+18,477634628,17499266792474872032,1765706303192859709,Russia,Novosibirsk,225,65,mac_os_sonoma,1,https://halltape.github.io/HalltapeRoadmapDE/Q...,[],internal,2025-12-14 22:36:50,2026-01-20 00:47:20.930323+03:00
1,7.602713e+18,415156879,16207496369939745913,1754300074367228741,Russia,Moscow,225,213,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,[],referral,2025-12-21 19:07:28,2026-01-20 00:47:20.930323+03:00
2,7.602717e+18,415156879,16207496369939745913,1754300074367228741,Russia,Moscow,225,213,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,[],direct,2025-12-21 19:07:43,2026-01-20 00:47:20.930323+03:00
3,7.602745e+18,284362479,16207496369939745913,1754300074367228741,Russia,Moscow,225,213,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,[],referral,2025-12-21 19:09:30,2026-01-20 00:47:20.930323+03:00
4,7.602749e+18,284362479,16207496369939745913,1754300074367228741,Russia,Moscow,225,213,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,[],direct,2025-12-21 19:09:46,2026-01-20 00:47:20.930323+03:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206134,5.537328e+18,617003555,8467176817481702272,1758443200444950432,Serbia,Novi Sad,180,105422,mac_os_sonoma,1,https://halltape.github.io/HalltapeRoadmapDE/Q...,[],internal,2025-09-21 14:33:49,2026-01-20 00:47:20.930323+03:00
206135,5.531662e+18,17947725,8361575273927386543,1757448274999299289,Russia,Saint Petersburg,225,2,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,[],direct,2025-09-21 08:33:35,2026-01-20 00:47:20.930323+03:00
206136,5.531666e+18,17947725,8361575273927386543,1757448274999299289,Russia,Saint Petersburg,225,2,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,[],direct,2025-09-21 08:33:50,2026-01-20 00:47:20.930323+03:00
206137,5.538760e+18,1035913508,8361575273927386543,1757448274999299289,Russia,NaN,225,0,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,[],direct,2025-09-21 16:04:53,2026-01-20 00:47:20.930323+03:00


In [17]:
hits_df.columns = [
    col if i > 13 else col[6:]
    for i, col in enumerate(hits_df.columns)
]
hits_df

,watchID,pageViewID,counterUserIDHash,clientID,regionCountry,regionCity,regionCountryID,regionCityID,operatingSystem,deviceCategory,URL,productEventType,lastTrafficSource,dateTime,updated_at
0,7.467877e+18,477634628,17499266792474872032,1765706303192859709,Russia,Novosibirsk,225,65,mac_os_sonoma,1,https://halltape.github.io/HalltapeRoadmapDE/Q...,[],internal,2025-12-14 22:36:50,2026-01-20 00:47:20.930323+03:00
1,7.602713e+18,415156879,16207496369939745913,1754300074367228741,Russia,Moscow,225,213,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,[],referral,2025-12-21 19:07:28,2026-01-20 00:47:20.930323+03:00
2,7.602717e+18,415156879,16207496369939745913,1754300074367228741,Russia,Moscow,225,213,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,[],direct,2025-12-21 19:07:43,2026-01-20 00:47:20.930323+03:00
3,7.602745e+18,284362479,16207496369939745913,1754300074367228741,Russia,Moscow,225,213,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,[],referral,2025-12-21 19:09:30,2026-01-20 00:47:20.930323+03:00
4,7.602749e+18,284362479,16207496369939745913,1754300074367228741,Russia,Moscow,225,213,windows11,1,https://halltape.github.io/HalltapeRoadmapDE/,[],direct,2025-12-21 19:09:46,2026-01-20 00:47:20.930323+03:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206134,5.537328e+18,617003555,8467176817481702272,1758443200444950432,Serbia,Novi Sad,180,105422,mac_os_sonoma,1,https://halltape.github.io/HalltapeRoadmapDE/Q...,[],internal,2025-09-21 14:33:49,2026-01-20 00:47:20.930323+03:00
206135,5.531662e+18,17947725,8361575273927386543,1757448274999299289,Russia,Saint Petersburg,225,2,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,[],direct,2025-09-21 08:33:35,2026-01-20 00:47:20.930323+03:00
206136,5.531666e+18,17947725,8361575273927386543,1757448274999299289,Russia,Saint Petersburg,225,2,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,[],direct,2025-09-21 08:33:50,2026-01-20 00:47:20.930323+03:00
206137,5.538760e+18,1035913508,8361575273927386543,1757448274999299289,Russia,NaN,225,0,windows10,1,https://halltape.github.io/HalltapeRoadmapDE/#...,[],direct,2025-09-21 16:04:53,2026-01-20 00:47:20.930323+03:00


In [18]:
hits_df.to_csv('./tmp/data_hits.csv', sep = '\t', index = False)